# First attempt to fir gradients maps to ASD fMRI data

In [ ]:
import argparse
import os.path as op
import os
import numpy as np
import nibabel as nib
import nilearn
from nilearn.connectome import ConnectivityMeasure
from nilearn import signal
import pandas as pd


In [2]:
bids_folder = "/mnt_01/ds-ASD"

In [ ]:
runs_per_sub_thresh = 2

In [ ]:
# Parameter definieren, fMRI Daten laden
def cleanTS(sub, fmriprep_confounds_include, bids_folder,
            TR=2.3, ses="00001", task="fmri", 
            runs=range(1,10),
            scrubbing=True, scrub_thresh=0.3,
            run_FD_filter=True, frames_per_run_thresh=104,
            bp_filtering=True, lower_bpf=0.01, upper_bpf=0.08):
    
fmriprep_folder = op.join(bids_folder, "derivatives", "fmriprep", f"sub-{sub}", f"ses-{ses}", "func")

clean_ts_runs = np.empty([0,0])
N_valid_runs = 0

for run in runs:
    try:
        # === 1. Load fMRI data (volume) ===
        bold_file = op.join(
            fmriprep_folder,
            f"sub-{sub}_ses-{ses}_task-{task}_run-{run}_space-MNI152NLin2009cAsym_desc-preproc_bold.nii"
        )
        print("Loading:", bold_file)
        bold_img = nib.load(bold_file)

        # Flatten volume -> time series (voxels × time)
        timeseries = masking.apply_mask(bold_img, 
                    op.join(fmriprep_folder,
                            f"sub-{sub}_ses-{ses}_task-{task}_run-{run}_space-MNI152NLin2009cAsym_desc-brain_mask.nii.gz"))
        # Load confounds
        confound_file = op.join(
                fmriprep_folder,
                f"sub-{sub}_ses-{ses}_task-{task}_run-{run}_desc-confounds_timeseries.tsv"
            )
            fmriprep_confounds = pd.read_table(confound_file)[fmriprep_confounds_include]
            fmriprep_confounds = fmriprep_confounds.bfill()
        
        # Scrubbing
        if scrubbing:
                fd = pd.read_table(confound_file)["framewise_displacement"]
                sample_mask = (fd < scrub_thresh).to_numpy()
                usable_frames = np.sum(sample_mask)
                if run_FD_filter and usable_frames < frames_per_run_thresh:
                    print(f"Run {run} has only {usable_frames} usable frames, skipping.")
                    continue
            else:
                sample_mask = None

            # Cleaning
            clean_ts = signal.clean(
                timeseries,
                confounds=fmriprep_confounds,
                sample_mask=sample_mask,
                t_r=TR,
                standardize="zscore_sample",
                low_pass=upper_bpf if bp_filtering else None,
                high_pass=lower_bpf if bp_filtering else None
            )
            # Concatenate runs
            if clean_ts_runs.size == 0:
                clean_ts_runs = clean_ts
            else:
                clean_ts_runs = np.vstack([clean_ts_runs, clean_ts])

            N_valid_runs += 1

        except Exception as e:
            print(f"Error in sub-{sub}, run-{run}: {e}")

    return clean_ts_runs, N_valid_runs 


In [ ]:
# Dateien
func_file = op.join(bids_folder, "derivatives", "fmriprep",
                    f"sub-{sub}", f"ses-{ses}", "func",
                    f"sub-{sub}_ses-{ses}_task-{task}_acq-4_rec-1_run-{run}_space-MNI152NLin2009cAsym_desc-preproc_bold.nii")

conf_file = func_file.replace("_desc-preproc_bold.nii", "_desc-confounds_timeseries.tsv")
mask_file = func_file.replace("_desc-preproc_bold.nii", "_desc-brain_mask.nii.gz")

print("fMRI file:", func_file)
print("Confounds file:", conf_file)
print("Mask file:", mask_file)

In [ ]:
def main

In [ ]:
# target folder for CM
import os.path as op

target_folder = op.join(bids_folder, "derivatives", "correlation_matrices")

import os
os.makedirs(target_folder,exist_ok=True)